# Phase 2 — Step 3: Parent-Child Context Reconstruction

## Objective
Solve the **context loss problem** introduced by Phase 1's Custom RBAC chunking strategy.

When PII was isolated into tiny fragments (e.g., a 55-character IBAN or a password line), semantic context was destroyed. A retriever can find the fragment, but the LLM has no idea which document, clause, or log entry it belongs to.

## Approach
1. **Parent Index**: Build a lookup table mapping each isolated PII chunk → its original "parent" paragraph (from the `fixed` chunking strategy), keyed by `(source_file, chunk_id)` composite key.
2. **Detection**: When the retriever returns a chunk, detect if it's an isolated PII fragment (via `contains_PII` metadata + small size heuristic ≤120 chars).
3. **RBAC-gated Merge**: Fetch the parent chunk and merge it with the child **only if** the user's `clearance_level ≥ parent.clearance_level` AND department matches. Otherwise, **drop** the isolated fragment (it degrades LLM output without context).
4. **Before/After Comparison**: Show exactly how context is restored for targeted PII queries.

In [1]:
# Cell 1 — Imports & Configuration
import json, time, re, os, textwrap
from dataclasses import dataclass, field
from typing import Optional
import numpy as np
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
import chromadb

BASE = os.path.dirname(os.path.abspath("__file__"))
RESULTS_DIR = os.path.join(BASE, "..", "..", "data", "results", "notebook_results")
CHUNK_FILE = os.path.join(RESULTS_DIR, "chunk_results.json")

ISOLATED_CHAR_THRESHOLD = 120
K = 5

print(f"Results dir: {os.path.abspath(RESULTS_DIR)}")
print("Step 3: Parent-Child Reconstruction — ready.")

C:\Users\arnau\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Results dir: D:\URV\TFG\ai-rag-context-auth-system\data\results\notebook_results
Step 3: Parent-Child Reconstruction — ready.


In [2]:
# Cell 2 — Load corpus + build parent index (composite key: source_file + chunk_id)

with open(CHUNK_FILE, "r", encoding="utf-8") as f:
    all_strategies = json.load(f)

# --- Load custom_rbac corpus ---
corpus: list[Document] = []
for src_file, chunks in all_strategies["custom_rbac"].items():
    for ch in chunks:
        meta = ch["metadata"].copy()
        meta.setdefault("contains_PII", False)
        meta.setdefault("sensitivity_types", [])
        corpus.append(Document(page_content=ch["page_content"], metadata=meta))
print(f"Custom RBAC corpus: {len(corpus)} chunks")

# --- Load fixed corpus (parent paragraphs) ---
fixed_docs: list[Document] = []
for src_file, chunks in all_strategies["fixed"].items():
    for ch in chunks:
        fixed_docs.append(Document(page_content=ch["page_content"], metadata=ch["metadata"]))
print(f"Fixed corpus (parents): {len(fixed_docs)} chunks")

# --- Build Parent Index with COMPOSITE KEY ---
# chunk_ids repeat across source files → use (source_file, chunk_id)
parent_index: dict[tuple[str, str], Document] = {}

for child in corpus:
    cm = child.metadata
    if not cm.get("contains_PII", False):
        continue
    child_text = child.page_content.strip()
    child_src = cm["source_file"]
    child_id = cm["chunk_id"]
    key = (child_src, child_id)

    best_parent: Optional[Document] = None
    best_len = float("inf")

    for fxd in fixed_docs:
        if fxd.metadata["source_file"] != child_src:
            continue
        if child_text in fxd.page_content:
            if len(fxd.page_content) < best_len:
                best_parent = fxd
                best_len = len(fxd.page_content)

    if best_parent is not None:
        parent_index[key] = best_parent
    else:
        # Fallback: highest word overlap from same source
        best_overlap = 0
        for fxd in fixed_docs:
            if fxd.metadata["source_file"] != child_src:
                continue
            child_words = set(child_text.lower().split())
            parent_words = set(fxd.page_content.lower().split())
            overlap = len(child_words & parent_words)
            if overlap > best_overlap:
                best_overlap = overlap
                best_parent = fxd
        if best_parent is not None and best_overlap >= 2:
            parent_index[key] = best_parent

print(f"\nParent index: {len(parent_index)} PII chunks mapped to parents\n")
for (src, cid), parent in parent_index.items():
    child_doc = next(d for d in corpus
                     if d.metadata["chunk_id"] == cid and d.metadata["source_file"] == src)
    cm = child_doc.metadata
    pm = parent.metadata
    stypes = cm.get("sensitivity_types", [])
    if isinstance(stypes, str):
        stypes = json.loads(stypes)
    print(f"  {src}::{cid} ({', '.join(stypes)}) [{len(child_doc.page_content)}ch, cl={cm['clearance_level']}]"
          f" → {pm['chunk_id']} [{len(parent.page_content)}ch, cl={pm['clearance_level']}]")

Custom RBAC corpus: 33 chunks
Fixed corpus (parents): 20 chunks

Parent index: 15 PII chunks mapped to parents

  Witty-QuickGuide-EN.pdf::custom_000 (email_personal) [17ch, cl=2] → fixed_001 [152ch, cl=0]
  distribution-contract-2026.docx::custom_000 (person_name) [250ch, cl=2] → fixed_000 [346ch, cl=2]
  distribution-contract-2026.docx::custom_004 (iban) [55ch, cl=3] → fixed_002 [310ch, cl=2]
  distribution-contract-2026.docx::custom_005 (swift) [15ch, cl=3] → fixed_002 [310ch, cl=2]
  server_logs_witty_backend.txt::custom_002 (ip_address) [21ch, cl=2] → fixed_000 [462ch, cl=2]
  server_logs_witty_backend.txt::custom_005 (password) [48ch, cl=3] → fixed_001 [286ch, cl=2]
  clients-and-billings.xlsx::custom_000 (person_name) [104ch, cl=3] → fixed_000 [433ch, cl=3]
  clients-and-billings.xlsx::custom_001 (email_personal) [104ch, cl=3] → fixed_000 [433ch, cl=3]
  clients-and-billings.xlsx::custom_002 (client_id) [104ch, cl=3] → fixed_000 [433ch, cl=3]
  clients-and-billings.xlsx::custom_

In [3]:
# Cell 3 — ChromaDB + BM25 Setup

print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="step3_pc", metadata={"hnsw:space": "cosine"})

texts = [d.page_content for d in corpus]

def sanitize(meta: dict) -> dict:
    return {k: (json.dumps(v) if isinstance(v, list) else v if isinstance(v, (str, int, float, bool)) else str(v))
            for k, v in meta.items()}

print("Computing embeddings...")
embs = embedding_model.embed_documents(texts)
collection.add(documents=texts, embeddings=embs,
               ids=[f"c{i:03d}" for i in range(len(corpus))],
               metadatas=[sanitize(d.metadata) for d in corpus])
print(f"ChromaDB: {collection.count()} docs")

def tokenize(t: str) -> list[str]:
    return re.findall(r"\w+", t.lower())

tok_corpus = [tokenize(d.page_content) for d in corpus]
bm25 = BM25Okapi(tok_corpus)
print(f"BM25: {len(tok_corpus)} docs")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing embeddings...


ChromaDB: 33 docs
BM25: 33 docs


In [4]:
# Cell 4 — Retrieval Functions + Intent Router (from Step 2)

@dataclass
class RR:
    document: Document
    score: float
    source_engine: str
    rank: int = 0

def ret_chroma(q: str, k: int = 5) -> list[RR]:
    qe = embedding_model.embed_query(q)
    r = collection.query(query_embeddings=[qe], n_results=k,
                         include=["documents", "metadatas", "distances"])
    return [RR(Document(page_content=r["documents"][0][i], metadata=r["metadatas"][0][i]),
               1.0 - r["distances"][0][i], "chroma", i + 1)
            for i in range(len(r["documents"][0]))]

def ret_bm25(q: str, k: int = 5) -> list[RR]:
    sc = bm25.get_scores(tokenize(q))
    top = np.argsort(sc)[::-1][:k]
    return [RR(corpus[i], float(sc[i]), "bm25", r + 1)
            for r, i in enumerate(top) if sc[i] > 0]

def ens_rrf(q: str, k: int = 5, alpha: float = 0.5, rk: int = 60) -> list[RR]:
    cr, br = ret_chroma(q, k * 2), ret_bm25(q, k * 2)
    sc, dm, sm = {}, {}, {}
    for r in cr:
        key = r.document.page_content
        sc[key] = sc.get(key, 0) + alpha * (1.0 / (rk + r.rank))
        dm[key] = r.document; sm.setdefault(key, []).append("chroma")
    for r in br:
        key = r.document.page_content
        sc[key] = sc.get(key, 0) + (1 - alpha) * (1.0 / (rk + r.rank))
        dm[key] = r.document; sm.setdefault(key, []).append("bm25")
    sk = sorted(sc, key=sc.get, reverse=True)[:k]
    return [RR(dm[key], sc[key], "ens(" + "+".join(sorted(set(sm[key]))) + ")", i + 1)
            for i, key in enumerate(sk)]

# Intent Router (Tier-Priority from Step 2 v2)
@dataclass
class IC:
    intent: str; alpha: float; patterns: list[str]; reasoning: str

T1 = [{"n": "password", "p": r"(?i)\b(?:password|contraseña|pwd|credential|override)\b"},
      {"n": "iban", "p": r"(?i)\bIBAN\b"}, {"n": "swift", "p": r"(?i)\bSWIFT\b"}]
T2 = [{"n": "client_id", "p": r"(?i)\bCLI-\d{3}\b"},
      {"n": "ip", "p": r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b"},
      {"n": "email", "p": r"(?i)\b(?:email|e-mail|correo)\s*(?:address|de)?"},
      {"n": "log", "p": r"(?i)\b(?:log|logs|error\s+log|server\s+log)\b"},
      {"n": "account", "p": r"(?i)\b(?:account\s+number|bank\s+account)\b"},
      {"n": "name", "p": r"\b[A-Z][a-záéíóúñ]+\s+(?:Gómez|Ruiz|Mendoza|Torres|Silva)\b"},
      {"n": "id_seek", "p": r"(?i)\b(?:identifier|code|codi|código|número)\b"}]
T3 = [{"n": "how", "p": r"(?i)^(?:how|com|cómo)\s"},
      {"n": "what", "p": r"(?i)^(?:what|què|qué)\s(?:is|are|és)\b"},
      {"n": "describe", "p": r"(?i)\b(?:summary|overview|resum|explain|describe)\b"},
      {"n": "perf", "p": r"(?i)\b(?:performance|revenue|budget|rendiment|ingressos)\b"},
      {"n": "product", "p": r"(?i)\b(?:product|feature|kit|timer|contents|specifications)\b"},
      {"n": "quarter", "p": r"(?i)\bQ[1-4]\b"},
      {"n": "topic", "p": r"(?i)\b(?:about|sobre|regarding|strategy|forecast)\b"}]

def classify(q: str) -> IC:
    t1 = [p["n"] for p in T1 if re.search(p["p"], q)]
    t2 = [p["n"] for p in T2 if re.search(p["p"], q)]
    t3 = [p["n"] for p in T3 if re.search(p["p"], q)]
    allp, exact = t1 + t2 + t3, t1 + t2
    if t1: return IC("exact_critical", 0.2, allp, f"TIER1: {t1}")
    if t2 and not t3: return IC("exact", 0.2, allp, f"Exact: {t2}")
    if t3 and not exact: return IC("conceptual", 0.8, allp, f"Conceptual: {t3}")
    if exact and t3:
        if len(exact) * 1.5 >= len(t3): return IC("mixed_exact_lean", 0.35, allp, "Mixed exact-lean")
        return IC("mixed_concept_lean", 0.65, allp, "Mixed concept-lean")
    return IC("default", 0.5, [], "No signal")

print("Retrieval + intent router loaded.")

Retrieval + intent router loaded.


In [5]:
# Cell 5 — Parent-Child Reconstruction Engine + PII Detection + RBAC

# --- PII Detection ---
PII_RE = {
    "email": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
    "iban": r"[A-Z]{2}\d{2}[\s]?[A-Z0-9]{4}[\s]?[\d]{4}[\s]?[\d]{4}[\s]?[\d]{4}",
    "ip_address": r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b",
    "password": r"(?i)(?:password|override[_ ]?password)\s*[:=]\s*['\"]?([^\s'\"]+)",
    "client_id": r"CLI-\d{3,}",
    "swift_code": r"SWIFT[:\s]*[A-Z]{4}[A-Z]{2}[A-Z0-9]{2,5}",
}

def det_pii(t: str) -> dict:
    return {n: m for n, p in PII_RE.items() if (m := re.findall(p, t))}

# --- Reconstruction ---
@dataclass
class ReconAction:
    chunk_id: str
    source_file: str
    original_text: str
    action: str          # KEPT | MERGED | DROPPED
    reason: str
    parent_chunk_id: Optional[str] = None
    merged_text: Optional[str] = None

def is_isolated_pii(doc: Document) -> bool:
    meta = doc.metadata
    contains_pii = meta.get("contains_PII", False)
    if isinstance(contains_pii, str):
        contains_pii = contains_pii.lower() == "true"
    if not contains_pii:
        return False
    return len(doc.page_content.strip()) <= ISOLATED_CHAR_THRESHOLD

def check_rbac(user_cl: int, user_dept: str, chunk_meta: dict) -> bool:
    chunk_cl = int(chunk_meta.get("clearance_level", 0))
    chunk_dept = chunk_meta.get("allowed_departments", "all")
    if chunk_cl > user_cl:
        return False
    if chunk_cl >= 2 and chunk_dept != "all" and user_dept != chunk_dept:
        return False
    return True

def reconstruct(results: list[RR], user_cl: int, user_dept: str
                ) -> tuple[list[RR], list[ReconAction]]:
    """Parent-Child Reconstruction with RBAC enforcement."""
    reconstructed: list[RR] = []
    actions: list[ReconAction] = []
    seen_texts: set[str] = set()

    for r in results:
        meta = r.document.metadata
        chunk_id = meta.get("chunk_id", "?")
        src_file = meta.get("source_file", "?")
        text = r.document.page_content

        # Case 1: Not isolated PII → keep
        if not is_isolated_pii(r.document):
            if text not in seen_texts:
                reconstructed.append(r)
                seen_texts.add(text)
                actions.append(ReconAction(chunk_id, src_file,
                    text[:80] + ("..." if len(text) > 80 else ""),
                    "KEPT", "Not an isolated PII fragment"))
            continue

        # Case 2: Isolated PII → look up parent via composite key
        key = (src_file, chunk_id)
        parent = parent_index.get(key)

        if parent is None:
            actions.append(ReconAction(chunk_id, src_file, text[:80],
                "DROPPED", "No parent found in index"))
            continue

        parent_meta = parent.metadata
        parent_text = parent.page_content

        if check_rbac(user_cl, user_dept, parent_meta):
            # RBAC passes → MERGE
            if parent_text not in seen_texts:
                merged_doc = Document(
                    page_content=parent_text,
                    metadata={**parent_meta,
                              "reconstructed_from": chunk_id,
                              "original_pii_types": meta.get("sensitivity_types", "[]"),
                              "reconstruction": "parent_child_merge"})
                reconstructed.append(RR(merged_doc, r.score, r.source_engine + "+parent", r.rank))
                seen_texts.add(parent_text)
                actions.append(ReconAction(chunk_id, src_file, text[:80],
                    "MERGED",
                    f"Parent RBAC OK (cl={parent_meta.get('clearance_level')}, dept={parent_meta.get('allowed_departments')})",
                    parent_meta.get("chunk_id"),
                    parent_text[:100] + ("..." if len(parent_text) > 100 else "")))
            else:
                actions.append(ReconAction(chunk_id, src_file, text[:80],
                    "KEPT", "Parent already in ranking (dedup)"))
        else:
            # RBAC fails → DROP
            actions.append(ReconAction(chunk_id, src_file, text[:80],
                "DROPPED",
                f"RBAC denied (parent cl={parent_meta.get('clearance_level')}, "
                f"dept={parent_meta.get('allowed_departments')} | "
                f"user cl={user_cl}, dept={user_dept})"))

    return reconstructed, actions

def audit_results(results: list[RR], user_cl: int, user_dept: str) -> dict:
    total_breaches = 0
    all_pii: set[str] = set()
    for r in results:
        meta = r.document.metadata
        chunk_cl = int(meta.get("clearance_level", 0))
        chunk_dept = meta.get("allowed_departments", "all")
        breach = (chunk_cl > user_cl) or (chunk_cl >= 2 and chunk_dept != "all" and user_dept != chunk_dept)
        if breach:
            total_breaches += 1
        all_pii.update(det_pii(r.document.page_content).keys())
    return {"breaches": total_breaches, "pii_types": sorted(all_pii),
            "security": "PASS" if total_breaches == 0 else "FAIL"}

print("Reconstruction engine + PII detection + RBAC loaded.")

Reconstruction engine + PII detection + RBAC loaded.


## Test Scenarios

| ID | Query | User Profile | Expected |
|----|-------|-------------|----------|
| T1 | Password query | Engineer cl=3 | **MERGE** — full log context restored |
| T2 | Password query | Sales cl=1 | **DROP** — clearance insufficient |
| T3 | IBAN query | Legal cl=3 | **MERGE** — contract clause restored |
| T4 | IBAN query | Engineer cl=2 | **DROP** — wrong department |
| T5 | Client emails | Sales cl=3 | **MERGE** — client rows with context |
| T6 | Client emails | Finance cl=3 | **DROP** — wrong department |
| T7 | IP address | Engineer cl=2 | **MERGE** — log entry restored |
| T8 | Public product | Public cl=0 | **KEPT** — no reconstruction needed |

In [6]:
# Cell 6 — Test Queries Definition

@dataclass
class TestCase:
    tid: str; query: str; user_cl: int; user_dept: str
    description: str; expected_action: str
    target_pii: list[str] = field(default_factory=list)

TESTS = [
    TestCase("T1", "What is the admin override password for the backend?",
             3, "engineering", "Password — authorized engineer", "MERGE", ["password"]),
    TestCase("T2", "What is the admin override password for the backend?",
             1, "sales", "Password — unauthorized sales intern", "DROP", ["password"]),
    TestCase("T3", "IBAN bank account number for payments",
             3, "legal", "IBAN — authorized legal counsel", "MERGE", ["iban", "swift_code"]),
    TestCase("T4", "IBAN bank account number for payments",
             2, "engineering", "IBAN — wrong department engineer", "DROP", ["iban", "swift_code"]),
    TestCase("T5", "email addresses of clients Laura Gomez and Marcos Ruiz",
             3, "sales", "Client emails — authorized sales manager", "MERGE", ["email", "client_id"]),
    TestCase("T6", "email addresses of clients Laura Gomez and Marcos Ruiz",
             3, "finance", "Client emails — wrong department finance", "DROP", ["email", "client_id"]),
    TestCase("T7", "What IP address caused the failed login attempt in server logs?",
             2, "engineering", "IP address — authorized engineer", "MERGE", ["ip_address"]),
    TestCase("T8", "What are the contents of the Witty Kit?",
             0, "all", "Public product info — no PII reconstruction", "KEPT", []),
]

print(f"{len(TESTS)} test cases defined.")

8 test cases defined.


In [7]:
# Cell 7 — Execute all tests with BEFORE/AFTER comparison

all_results = []

for tc in TESTS:
    print("=" * 90)
    print(f"  {tc.tid}: {tc.description}")
    print(f"  Query: \"{tc.query}\"")
    print(f"  User: cl={tc.user_cl}, dept={tc.user_dept} | Expected: {tc.expected_action}")
    print("=" * 90)

    ic = classify(tc.query)
    print(f"\n  Intent: {ic.intent} (α={ic.alpha})")

    t0 = time.time()
    raw_results = ens_rrf(tc.query, k=K, alpha=ic.alpha)
    lat_ret = (time.time() - t0) * 1000

    # --- BEFORE ---
    before_audit = audit_results(raw_results, tc.user_cl, tc.user_dept)
    print(f"\n  ── BEFORE Reconstruction ({len(raw_results)} chunks, {lat_ret:.0f}ms) ──")
    for i, r in enumerate(raw_results):
        m = r.document.metadata
        pii = det_pii(r.document.page_content)
        pii_s = ", ".join(pii.keys()) if pii else "-"
        iso = " [ISOLATED]" if is_isolated_pii(r.document) else ""
        txt = r.document.page_content[:65].replace("\n", " ")
        print(f"    #{i+1} {m.get('chunk_id','?'):12s} cl={m.get('clearance_level',0)} "
              f"dept={str(m.get('allowed_departments','all')):12s} PII={pii_s:20s}{iso}")
        print(f"       \"{txt}...\"")
    print(f"  Audit: breaches={before_audit['breaches']}, PII={before_audit['pii_types']}, "
          f"sec={before_audit['security']}")

    # --- RECONSTRUCT ---
    t1 = time.time()
    recon_results, actions = reconstruct(raw_results, tc.user_cl, tc.user_dept)
    lat_rec = (time.time() - t1) * 1000

    print(f"\n  ── Reconstruction Actions ({lat_rec:.1f}ms) ──")
    for a in actions:
        if a.action == "MERGED":
            print(f"    ✓ {a.action} {a.source_file}::{a.chunk_id} → parent {a.parent_chunk_id}")
            print(f"      Child:  \"{a.original_text}\"")
            print(f"      Parent: \"{a.merged_text}\"")
        elif a.action == "DROPPED":
            print(f"    ✗ {a.action} {a.source_file}::{a.chunk_id}")
            print(f"      Text:   \"{a.original_text}\"")
            print(f"      Reason: {a.reason}")
        else:
            print(f"    · {a.action} {a.source_file}::{a.chunk_id}")

    # --- AFTER ---
    after_audit = audit_results(recon_results, tc.user_cl, tc.user_dept)
    print(f"\n  ── AFTER Reconstruction ({len(recon_results)} chunks) ──")
    for i, r in enumerate(recon_results):
        m = r.document.metadata
        pii = det_pii(r.document.page_content)
        pii_s = ", ".join(pii.keys()) if pii else "-"
        tag = " [RECONSTRUCTED]" if m.get("reconstruction") else ""
        txt = r.document.page_content[:65].replace("\n", " ")
        print(f"    #{i+1} {m.get('chunk_id','?'):12s} cl={m.get('clearance_level',0)} "
              f"dept={str(m.get('allowed_departments','all')):12s} PII={pii_s:20s}{tag}")
        print(f"       \"{txt}...\"")
    print(f"  Audit: breaches={after_audit['breaches']}, PII={after_audit['pii_types']}, "
          f"sec={after_audit['security']}")

    # Classify result
    merge_n = sum(1 for a in actions if a.action == "MERGED")
    drop_n = sum(1 for a in actions if a.action == "DROPPED")
    actual = "MERGE" if merge_n > 0 else ("DROP" if drop_n > 0 else "KEPT")
    match = "PASS" if actual == tc.expected_action else "FAIL"
    db = after_audit["breaches"] - before_audit["breaches"]
    print(f"\n  >> {actual} (expected {tc.expected_action}) → {match} | "
          f"Δbreaches={db:+d}, Δchunks={len(recon_results)-len(raw_results):+d}")

    all_results.append({
        "tid": tc.tid, "query": tc.query, "user_cl": tc.user_cl, "user_dept": tc.user_dept,
        "description": tc.description, "intent": ic.intent, "alpha": ic.alpha,
        "before_chunks": len(raw_results), "after_chunks": len(recon_results),
        "before_breaches": before_audit["breaches"], "after_breaches": after_audit["breaches"],
        "before_pii": ", ".join(before_audit["pii_types"]),
        "after_pii": ", ".join(after_audit["pii_types"]),
        "before_sec": before_audit["security"], "after_sec": after_audit["security"],
        "merges": merge_n, "drops": drop_n,
        "expected": tc.expected_action, "actual": actual, "match": match,
        "lat_retrieval_ms": round(lat_ret, 1), "lat_recon_ms": round(lat_rec, 1),
    })
    print()

  T1: Password — authorized engineer
  Query: "What is the admin override password for the backend?"
  User: cl=3, dept=engineering | Expected: MERGE

  Intent: exact_critical (α=0.2)

  ── BEFORE Reconstruction (5 chunks, 8ms) ──
    #1 custom_005   cl=3 dept=engineering  PII=password             [ISOLATED]
       "override password: 'witty_admin_override_2026!$'..."
    #2 custom_003   cl=2 dept=engineering  PII=-                   
       "[2026-03-05 11:05:00] ERROR: Failed login attempt for user 'admin..."
    #3 custom_001   cl=0 dept=all          PII=-                   
       "ENG QUICK GUIDE Witty Manager Software The USB stick contains the..."
    #4 custom_004   cl=2 dept=engineering  PII=-                   
       "[2026-03-05 11:12:33] CRITICAL: System alert. The backup server r..."
    #5 custom_002   cl=0 dept=all          PII=-                   
       "Documentation The Witty Kit user manual is stored on the USB stic..."
  Audit: breaches=0, PII=['password'], sec=PA

In [8]:
# Cell 8 — Summary + Export

df = pd.DataFrame(all_results)

print("=" * 90)
print("  STEP 3: PARENT-CHILD RECONSTRUCTION — SUMMARY")
print("=" * 90)

cols = ["tid", "description", "before_breaches", "after_breaches", "merges", "drops",
        "expected", "actual", "match"]
print("\n", df[cols].to_string(index=False))

total = len(df)
passed = (df["match"] == "PASS").sum()
avg_b_before = df["before_breaches"].mean()
avg_b_after = df["after_breaches"].mean()
reduction = avg_b_before - avg_b_after

print(f"\n--- Aggregate ---")
print(f"  Accuracy:        {passed}/{total} ({passed/total*100:.0f}%)")
print(f"  Total merges:    {df['merges'].sum()}")
print(f"  Total drops:     {df['drops'].sum()}")
print(f"  Avg breaches BEFORE: {avg_b_before:.2f}")
print(f"  Avg breaches AFTER:  {avg_b_after:.2f}")
print(f"  Breach reduction:    {reduction:+.2f} ({reduction/max(avg_b_before,0.01)*100:+.0f}%)")

for label, subset in [("Authorized (MERGE)", df[df["expected"]=="MERGE"]),
                       ("Unauthorized (DROP)", df[df["expected"]=="DROP"])]:
    if len(subset) > 0:
        print(f"\n  {label}:")
        print(f"    Before: {subset['before_breaches'].mean():.2f} avg breaches")
        print(f"    After:  {subset['after_breaches'].mean():.2f} avg breaches")
        sec_before = (subset["before_sec"] == "PASS").sum()
        sec_after = (subset["after_sec"] == "PASS").sum()
        print(f"    Security PASS: {sec_before} → {sec_after}")

# Export CSV
csv_path = os.path.join(RESULTS_DIR, "ph2_step3_reconstruction_results.csv")
df.to_csv(csv_path, index=False)
print(f"\nExported: {csv_path}")

# Export JSON
json_export = {
    "step": "Phase 2 - Step 3: Parent-Child Reconstruction",
    "config": {"isolated_char_threshold": ISOLATED_CHAR_THRESHOLD, "k": K,
               "parent_index_size": len(parent_index), "corpus_size": len(corpus)},
    "parent_index": {
        f"{src}::{cid}": {
            "child_text": next(d.page_content for d in corpus
                               if d.metadata["chunk_id"]==cid and d.metadata["source_file"]==src)[:80],
            "parent_chunk_id": pdoc.metadata["chunk_id"],
            "parent_cl": pdoc.metadata["clearance_level"],
            "parent_dept": pdoc.metadata["allowed_departments"],
            "parent_chars": len(pdoc.page_content),
        }
        for (src, cid), pdoc in parent_index.items()
    },
    "results": all_results,
}
json_path = os.path.join(RESULTS_DIR, "ph2_step3_reconstruction_detail.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_export, f, indent=2, ensure_ascii=False)
print(f"Exported: {json_path}")
print("\nStep 3 complete. Awaiting approval before Step 4.")

  STEP 3: PARENT-CHILD RECONSTRUCTION — SUMMARY

 tid                                 description  before_breaches  after_breaches  merges  drops expected actual match
 T1              Password — authorized engineer                0               0       1      0    MERGE  MERGE  PASS
 T2        Password — unauthorized sales intern                3               2       0      1     DROP   DROP  PASS
 T3             IBAN — authorized legal counsel                3               3       1      0    MERGE  MERGE  PASS
 T4            IBAN — wrong department engineer                3               2       0      1     DROP   DROP  PASS
 T5    Client emails — authorized sales manager                2               2       1      0    MERGE  MERGE  PASS
 T6    Client emails — wrong department finance                3               0       0      3     DROP   DROP  PASS
 T7            IP address — authorized engineer                2               2       1      0    MERGE  MERGE  PASS
 T8 Pu